# Nikolaisen2022 bin 01: small filtered Plag STL files

This notebook processes the filtered Plag production set from the second sorted file-size bin. It reports every mesh, while PyVista display is capped by `DISPLAY_LIMIT` to avoid huge saved notebook outputs.

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd()
if not (REPO / "src").exists() and (REPO.parent / "src").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

from stl2fem.datasets import add_nikolaisen_stl_metadata, assign_size_bins, nikolaisen_inventory
from stl2fem.visualize import display_volume_mesh
from stl2fem.conversion import DEFAULT_BRUTE_FORCE_MAX_CELLS_PER_AXIS
from stl2fem.workflow import DEFAULT_MESH_STRATEGIES, DEFAULT_MESH_TIMEOUT_SECONDS, process_nikolaisen_size_bin

DATASET_ROOT = REPO / "data" / "Nikolaisen2022"
OUTPUT_ROOT = REPO / "data" / "Nikolaisen2022_merrill_msh"
BIN_INDEX = 1
N_BINS = 4
INPUT_UNIT = "um"
TARGET_EDGE_LENGTH_M = 9e-9
PHASES = ("PLAG",)
MAX_EVSD_UM = 1.0
SAVE_NATIVE_MESH = False
MESH_TIMEOUT_SECONDS = DEFAULT_MESH_TIMEOUT_SECONDS
MESH_STRATEGIES = DEFAULT_MESH_STRATEGIES
BRUTEFORCE_MAX_CELLS_PER_AXIS = DEFAULT_BRUTE_FORCE_MAX_CELLS_PER_AXIS
OVERWRITE = False
MAX_MESHES = None
DISPLAY_LIMIT = 8

In [ ]:
inventory = add_nikolaisen_stl_metadata(
    nikolaisen_inventory(DATASET_ROOT, phases=PHASES),
    dataset_root=DATASET_ROOT,
)
inventory = inventory[inventory["metadata_evsd_um"] < MAX_EVSD_UM].copy()
inventory = assign_size_bins(inventory, n_bins=N_BINS)
subset = inventory[inventory["size_bin_index"] == BIN_INDEX]
print(f"Meshes in this notebook: {len(subset)}")
subset[[
    "particle_id", "phase", "stl_format", "metadata_evsd_um",
    "stl_size_mib", "size_bin", "source_path",
]]

In [ ]:
results = process_nikolaisen_size_bin(
    DATASET_ROOT,
    OUTPUT_ROOT,
    bin_index=BIN_INDEX,
    n_bins=N_BINS,
    input_unit=INPUT_UNIT,
    target_edge_length_m=TARGET_EDGE_LENGTH_M,
    phases=PHASES,
    max_evsd_um=MAX_EVSD_UM,
    save_native_mesh=SAVE_NATIVE_MESH,
    mesh_timeout_seconds=MESH_TIMEOUT_SECONDS,
    mesh_strategies=MESH_STRATEGIES,
    bruteforce_max_cells_per_axis=BRUTEFORCE_MAX_CELLS_PER_AXIS,
    overwrite=OVERWRITE,
    max_meshes=MAX_MESHES,
)

columns = [
    "particle_id", "phase", "status", "metadata_evsd_um", "stl_size_mib",
    "merrill_msh_size_bytes", "merrill_msh_size_mib",
    "input_unit", "input_scale_to_meters", "target_edge_length_m", "target_edge_length_native",
    "edge_length_median", "edge_length_p95", "edge_length_median_m", "edge_length_p95_m",
    "n_nodes", "n_tets", "volume_min", "scaled_jacobian_min",
    "radius_ratio_max", "estimated_memory_human", "msh_path", "merrill_msh_path",
]
available = [column for column in columns if column in results.columns]
results[available]

In [ ]:
ok = results[results["status"] == "ok"].sort_values("merrill_msh_size_bytes")
display_count = len(ok) if DISPLAY_LIMIT is None else min(DISPLAY_LIMIT, len(ok))
print(f"Displaying {display_count} of {len(ok)} converted meshes. Increase DISPLAY_LIMIT to inspect more.")

for _, row in ok.head(display_count).iterrows():
    print(f"{row['particle_id']}: {row['n_nodes']} nodes, {row['n_tets']} tets, {row['estimated_memory_human']} estimated Merrill.jl memory")
    if row.get("msh_path"):
        print(f"  native diagnostic mesh: {row['msh_path']}")
    print(f"  Merrill meters: {row.get('merrill_msh_path', '')}")
    display_volume_mesh(row["merrill_msh_path"], show_edges=True, opacity=0.35)